In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, from_json, col

# Schema for Kafka messages (assuming JSON format)
schema = StructType() \
    .add("txn_id", "long") \
    .add("account_id", "long") \
    .add("amount", "double") \
    .add("transaction_type", "string") \
    .add("location", "string")

# For real Kafka integration, replace 'rate' with Kafka source
# Uncomment below for Kafka:
# stream_df = spark.readStream \
#     .format("kafka") \
#     .option("kafka.bootstrap.servers", "your-kafka-server:9092") \
#     .option("subscribe", "banking-transactions") \
#     .option("startingOffsets", "latest") \
#     .load() \
#     .select(from_json(col("value").cast("string"), schema).alias("data")) \
#     .select("data.*")

# Simulated stream for demo
stream_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

In [0]:
from pyspark.sql.functions import expr

txn_stream = stream_df.select(
    expr("value as txn_id"),
    expr("value % 1000 as account_id"),
    expr("rand() * 200000 as amount"),
    expr("'DEBIT' as transaction_type"),
    expr("'India' as location"),
    current_timestamp().alias("timestamp")
)

In [0]:
txn_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/tmp/checkpoints/bronze") \
    .toTable("banking.banking.banking_bronze_transactions")